In [ ]:
from model import Tokenisation, Transformer
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import math

# Determine optimal floating-point precision
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    target_dtype = torch.bfloat16
    print("Using BF16 (Recommended)")
else:
    target_dtype = torch.float16
    print("Using FP16")

Using BF16 (Recommended)


# Learning from **Children Stories**

In [2]:
tokeniser = Tokenisation()

Using Qwen/Qwen2.5-0.5B Tokeniser


In [3]:
def train_children_stories(model: Transformer, train_dataset, 
          epochs: int = 3, batch_size: int = 16, lr: float = 1e-3, dtype: torch.dtype = target_dtype): 

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
    model.to(device) 
    # model.train() 

    print("Total training samples:", len(train_dataset), '\n') 
    print("Using device :", device, '\n') 

    train_loader = DataLoader( 
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        drop_last=True, 
    ) 

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr) 
    criterion = nn.CrossEntropyLoss() 

    # GradScaler is required for FP16 to prevent underflow; disabled automatically for BF16/CPU 
    scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16 and device.type == "cuda")) 

    try: 

        for epoch in range(epochs): 
            total_loss = 0.0 

            # Stores losses for the current 100-batch window 
            moving_losses = [] 

            for batch, (x, y) in enumerate(train_loader): 

                x, y = x.to(device), y.to(device) 

                optimizer.zero_grad() 

                # 1. Mixed Precision Forward Pass 
                with torch.autocast(device_type=device.type, dtype=dtype, enabled=(device.type == "cuda")): 
                    logits, _ = model(x, return_attentions=False) 
                    loss = criterion( 
                        logits.view(-1, logits.size(-1)), 
                        y.view(-1) 
                    ) 

                # 2. Scaled Backward Pass 
                scaler.scale(loss).backward() 

                # 3. Unscale gradients before clipping 
                scaler.unscale_(optimizer) 
                total_norm = torch.nn.utils.clip_grad_norm_( 
                    model.parameters(), 
                    max_norm=1.0 
                ) 


                print(f"Grad Norm: {total_norm:.4f}") 
                
                # 4. Step Optimizer and Update Scaler 
                scaler.step(optimizer) 
                scaler.update() 

                loss_value = loss.item() 
                total_loss += loss_value 

                # Add current batch loss to moving-average window 
                moving_losses.append(loss_value) 

                # Calculate 100-batch moving average 
                if len(moving_losses) > 50: 
                    moving_losses.pop(0) 

                moving_avg = sum(moving_losses) / len(moving_losses) 

                progress = (batch + 1) / len(train_loader) * 100 

                print( 
                    f"Epoch {epoch+1} | " 
                    f"Batch {batch+1}/{len(train_loader)} | " 
                    f"Progress: {progress:.2f}% | " 
                    f"Loss {loss_value:.4f} | " 
                    f"50-Batch Avg {moving_avg:.4f}" 
                ) 

            avg_loss = total_loss / len(train_loader) 

            print(f"== Epoch {epoch+1} done | " f"Avg loss {avg_loss:.4f} | " f"PPL {math.exp(avg_loss):.2f} ==") 

        # Completion of Training ================================================== 
        print("Saving current model parameters...") 
        
        torch.save( 
            model.state_dict(),  
            "ChildrenStores_transformer.pth" 
        ) 

        print("Model saved successfully! (ChildrenStories_transformer.pth)") 

        return model 

    # KeyBoard Interrupt Error ================================================== 
    except KeyboardInterrupt: 

        print("\n\nTraining interrupted!") 
        
        print("Saving current model parameters...") 

        torch.save( 
            model.state_dict(),  
            "ChildrenStories_transformer.pth" 
        ) 

        print("Model saved successfully! (ChildrenStories_transformer.pth)") 

        return model

In [4]:
import torch
from torch.utils.data import Dataset
from datasets import load_dataset
from datasets import load_from_disk



class ChildrenStoriesDataset(Dataset):

    def __init__(self, tokenizer, seq_len=128, num_stories=10000):

        self.tokenizer = tokenizer
        self.seq_len = seq_len

        print("Loading Children Stories dataset...")

        dataset = load_from_disk("data/Children-Stories-Collection")
        dataset = dataset["train"].select(range(min(num_stories, len(dataset["train"]))))


        print("Stories loaded: ", len(dataset))

        self.samples = []

        for story in dataset["text"]:

            # Tokenize one complete story
            story_tokens = tokenizer.encode(story, add_special_tokens=False)

            # Ignore stories that are too short
            if len(story_tokens) < seq_len + 1:
                continue

            # Create training samples WITHOUT crossing story boundaries
            for i in range(0, len(story_tokens) - seq_len, seq_len):

                x = story_tokens[i:i + seq_len]
                y = story_tokens[i + 1:i + seq_len + 1]

                # Repeat this training sample 10 times
                for _ in range(10):
                    self.samples.append((
                        torch.tensor(x, dtype=torch.long),
                        torch.tensor(y, dtype=torch.long)
                    ))

        print("Total training samples:", len(self.samples))


    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):
        return self.samples[idx]

In [5]:
train_dataset = ChildrenStoriesDataset(tokeniser.tokenizer, seq_len=150)

Loading Children Stories dataset...
Stories loaded:  10000
Total training samples: 219910


In [22]:
model = Transformer(d_model=512, d_k=64, h=8)

model.load_state_dict(
    torch.load("transformer.pth")
)

Using Qwen/Qwen2.5-0.5B Tokeniser
Total Trainable Parameters : 174,621,809 (174.62M)
Parameters Stored on GPU    : 0 (0.00M)


<All keys matched successfully>

In [23]:
model = train_children_stories(model, train_dataset, epochs=4, batch_size=20, lr=1e-3)

Total training samples: 219910 

Using device : cuda 

Grad Norm: 3.4816
Epoch 1 | Batch 1/10995 | Progress: 0.01% | Loss 6.6815 | 50-Batch Avg 6.6815
Grad Norm: 60.0146
Epoch 1 | Batch 2/10995 | Progress: 0.02% | Loss 19.6157 | 50-Batch Avg 13.1486
Grad Norm: 17.3669
Epoch 1 | Batch 3/10995 | Progress: 0.03% | Loss 12.3618 | 50-Batch Avg 12.8864
Grad Norm: 6.2224
Epoch 1 | Batch 4/10995 | Progress: 0.04% | Loss 10.0653 | 50-Batch Avg 12.1811
Grad Norm: 3.9519
Epoch 1 | Batch 5/10995 | Progress: 0.05% | Loss 7.9475 | 50-Batch Avg 11.3344
Grad Norm: 15.2152
Epoch 1 | Batch 6/10995 | Progress: 0.05% | Loss 8.7871 | 50-Batch Avg 10.9098
Grad Norm: 42.1045
Epoch 1 | Batch 7/10995 | Progress: 0.06% | Loss 13.2415 | 50-Batch Avg 11.2429
Grad Norm: 13.7896
Epoch 1 | Batch 8/10995 | Progress: 0.07% | Loss 9.7413 | 50-Batch Avg 11.0552
Grad Norm: 4.7823
Epoch 1 | Batch 9/10995 | Progress: 0.08% | Loss 7.9073 | 50-Batch Avg 10.7055


Training interrupted!
Saving current model parameters...
Model

In [ ]:
print("Tokenizer vocab size:", tokeniser.vocab_size)
print("Model vocab size:", model.InputPositionalEmbedding.token_embedding.num_embeddings)

Tokenizer vocab size: 50257
Model vocab size: 50257


# Training from the **TinyStories** Dataset from `HuggingFace`

In [24]:
tokeniser = Tokenisation()

Using Qwen/Qwen2.5-0.5B Tokeniser


In [25]:
def train(model: Transformer, train_dataset,
          epochs: int = 3, batch_size: int = 16, lr: float = 1e-3, dtype: torch.dtype = target_dtype):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    # model.train()

    print("Total training samples:", len(train_dataset), '\n')
    print("Using device :", device, '\n')

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # GradScaler is required for FP16 to prevent underflow; disabled automatically for BF16/CPU
    scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16 and device.type == "cuda"))

    try:

        for epoch in range(epochs):
            total_loss = 0.0

            # Stores losses for the current 100-batch window
            moving_losses = []

            for batch, (x, y) in enumerate(train_loader):

                x, y = x.to(device), y.to(device)

                optimizer.zero_grad()

                # 1. Mixed Precision Forward Pass
                with torch.autocast(device_type=device.type, dtype=dtype, enabled=(device.type == "cuda")):
                    logits, _ = model(x, return_attentions=False)
                    loss = criterion(
                        logits.view(-1, logits.size(-1)),
                        y.view(-1)
                    )

                # 2. Scaled Backward Pass
                scaler.scale(loss).backward()

                # 3. Unscale gradients before clipping
                scaler.unscale_(optimizer)
                total_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )


                print(f"Grad Norm: {total_norm:.4f}")
                
                # 4. Step Optimizer and Update Scaler
                scaler.step(optimizer)
                scaler.update()

                loss_value = loss.item()
                total_loss += loss_value

                # Add current batch loss to moving-average window
                moving_losses.append(loss_value)

                # Calculate 100-batch moving average
                if len(moving_losses) > 50:
                    moving_losses.pop(0)

                moving_avg = sum(moving_losses) / len(moving_losses)

                progress = (batch + 1) / len(train_loader) * 100

                print(
                    f"Epoch {epoch+1} | "
                    f"Batch {batch+1}/{len(train_loader)} | "
                    f"Progress: {progress:.2f}% | "
                    f"Loss {loss_value:.4f} | "
                    f"50-Batch Avg {moving_avg:.4f}"
                )

            avg_loss = total_loss / len(train_loader)

            print(f"== Epoch {epoch+1} done | " f"Avg loss {avg_loss:.4f} | " f"PPL {math.exp(avg_loss):.2f} ==")

    except KeyboardInterrupt:

        print("\n\nTraining interrupted!")
        
        print("Saving current model parameters...")

        torch.save(
            model.state_dict(), 
            "transformer.pth"
        )

        print("Model saved successfully!")

        return model

    return model

In [26]:
from datasets import load_from_disk

# Point it to the folder containing the .arrow files
ds = load_from_disk("./data/tinystories")

small_ds = ds["train"].select(range(100000))
# small_ds = ds["train"]

def tokenize_function(batch):
    return tokeniser.tokenizer(
        batch["text"],
        truncation=False,
        return_attention_mask=False
    )

tokenized_ds = small_ds.map(
    tokenize_function,
    batched=True,
    batch_size=64,
    remove_columns=["text"]
)

In [27]:
from torch.utils.data import Dataset

class TinyStoriesDataset(Dataset):

    def __init__(self, tokenized_ds, seq_len):
        self.seq_len = seq_len
        self.examples = []

        for example in tokenized_ds:
            tokens = example["input_ids"]

            # Create chunks from this story
            for i in range(0, len(tokens) - seq_len, seq_len):
                chunk = tokens[i:i + seq_len + 1]

                if len(chunk) == seq_len + 1:
                    self.examples.append(chunk)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        chunk = self.examples[idx]

        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)

        return x, y

In [28]:
train_dataset = TinyStoriesDataset(tokenized_ds, seq_len=150)
# X, Y = train_dataset[0]

In [29]:
model = Transformer(d_model=512, d_k=64, h=8)

model.load_state_dict(
    torch.load("transformer.pth")
)

Using Qwen/Qwen2.5-0.5B Tokeniser
Total Trainable Parameters : 174,621,809 (174.62M)
Parameters Stored on GPU    : 0 (0.00M)


<All keys matched successfully>

In [30]:
model = train(model, train_dataset, epochs=4, batch_size=10, lr=1e-7)

Total training samples: 96699 

Using device : cuda 

Grad Norm: 1.2709
Epoch 1 | Batch 1/9669 | Progress: 0.01% | Loss 2.0635 | 50-Batch Avg 2.0635
Grad Norm: 1.3275
Epoch 1 | Batch 2/9669 | Progress: 0.02% | Loss 1.8856 | 50-Batch Avg 1.9745
Grad Norm: 1.2703
Epoch 1 | Batch 3/9669 | Progress: 0.03% | Loss 1.9445 | 50-Batch Avg 1.9645
Grad Norm: 1.2798
Epoch 1 | Batch 4/9669 | Progress: 0.04% | Loss 1.8819 | 50-Batch Avg 1.9439
Grad Norm: 1.3345
Epoch 1 | Batch 5/9669 | Progress: 0.05% | Loss 2.0477 | 50-Batch Avg 1.9646
Grad Norm: 1.3329
Epoch 1 | Batch 6/9669 | Progress: 0.06% | Loss 2.3049 | 50-Batch Avg 2.0214
Grad Norm: 1.2713
Epoch 1 | Batch 7/9669 | Progress: 0.07% | Loss 1.9385 | 50-Batch Avg 2.0095
Grad Norm: 1.2430
Epoch 1 | Batch 8/9669 | Progress: 0.08% | Loss 1.8607 | 50-Batch Avg 1.9909
Grad Norm: 1.2856
Epoch 1 | Batch 9/9669 | Progress: 0.09% | Loss 2.0281 | 50-Batch Avg 1.9950
Grad Norm: 1.3251
Epoch 1 | Batch 10/9669 | Progress: 0.10% | Loss 2.1069 | 50-Batch Avg 2.

# Learning from **Wizard of Oz** book

In [ ]:
# Cleaning the book

import re

# Load the book
with open(r"data\wizarfofoz.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Replace all whitespace (newlines, tabs, multiple spaces) with one space
text = re.sub(r'\s+', ' ', text).strip()

# Save the cleaned book
with open(r"data\wizarfofoz.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("Original characters:", len(text))
print("Cleaned book saved successfully!")

Original characters: 203335
Cleaned book saved successfully!


In [37]:
with open(r"data\wizarfofoz.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Words:", len(text.split()))

Words: 39109


In [38]:
tokens = tokeniser.encode(text)

print("Number of tokens:", len(tokens))
print(tokens[:20])

Number of tokens: 47901
[35, 269, 28571, 12163, 304, 279, 34346, 315, 279, 2244, 20148, 548, 1310, 550, 11, 448, 50421, 17599, 11, 879]


In [39]:
import torch
from torch.utils.data import Dataset

class BookDataset(Dataset):

    def __init__(self, tokens, seq_len=128):
        self.tokens = torch.tensor(tokens, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):

        x = self.tokens[idx : idx + self.seq_len]
        y = self.tokens[idx + 1 : idx + self.seq_len + 1]

        return x, y

In [47]:
train_dataset = BookDataset(tokens, seq_len=150)

In [48]:
model = Transformer(d_model=512, d_k=64, h=8)
model.load_state_dict(torch.load("transformer.pth"))

Using Qwen/Qwen2.5-0.5B Tokeniser
Total Trainable Parameters : 174,621,809 (174.62M)
Parameters Stored on GPU    : 0 (0.00M)


<All keys matched successfully>

In [ ]:
model = train(model, train_dataset, epochs=4, batch_size=10, lr=1e-4)

In [23]:
prompt = "Dorothy lived in the "
model.generate(prompt)

'Dorothy lived in the  was � Dorothy�� were � said it the a� Lion the at she “ the, of.� for it., with they and, Scare of of of Dorothy. but, the on was them but him of that and not with'

In [9]:
print(type(model))
print(next(model.parameters()).device)

<class 'main.Transformer'>
cpu
